In [ ]:
import os
import pandas as pd
import re
from datasets import Dataset, load_dataset

from utils.bq import get_client, query_to_df
from utils.hub_card import push_dataset_card

client, PROJECT_NAME = get_client()

In [ ]:
# Medication records for admitted patients covering two phases:
#   1. ED boarding phase: Pyxis dispenses during the ED stay (in_er = True)
#   2. Hospital ward phase: eMAR administrations from ED departure up to ICU transfer (in_er = False)
#
# ICU gate uses icu.icustays (not transfers) — only patients under actual ICU clinical management.
# eMAR event_txt values mapped to four discrete actions:
#   administer_medication, start_medication, stop_medication, rate_change.
# Flush, topical, and documentation events excluded.

query_admitted_meds = """
WITH icu_times AS (
  SELECT
    hadm_id,
    CAST(MIN(intime) AS TIMESTAMP) AS icu_intime
  FROM `physionet-data.mimiciv_3_1_icu.icustays`
  GROUP BY hadm_id
),

pyxis_part AS (
  SELECT
    e.subject_id,
    e.stay_id,
    e.hadm_id,
    CAST(p.charttime AS TIMESTAMP) AS pyxis_charttime,
    p.med_rn        AS pyxis_med_rn,
    p.name          AS pyxis_medication,
    CAST(NULL AS STRING)    AS emar_id,
    CAST(NULL AS TIMESTAMP) AS emar_charttime,
    CAST(NULL AS STRING)    AS emar_medication,
    CAST(NULL AS STRING)    AS event_txt,
    CAST(NULL AS TIMESTAMP) AS scheduletime,
    TRUE            AS in_er
  FROM `physionet-data.mimiciv_ed.edstays` e
  JOIN `physionet-data.mimiciv_ed.pyxis` p
    ON e.stay_id = p.stay_id
  WHERE (e.disposition = 'ADMITTED' OR e.hadm_id IS NOT NULL)
    AND p.charttime BETWEEN e.intime AND e.outtime
),

emar_part AS (
  SELECT
    e.subject_id,
    e.stay_id,
    em.hadm_id,
    CAST(NULL AS TIMESTAMP) AS pyxis_charttime,
    CAST(NULL AS INT64)     AS pyxis_med_rn,
    CAST(NULL AS STRING)    AS pyxis_medication,
    em.emar_id,
    CAST(em.charttime AS TIMESTAMP) AS emar_charttime,
    em.medication           AS emar_medication,
    em.event_txt,
    CAST(em.scheduletime AS TIMESTAMP) AS scheduletime,
    FALSE           AS in_er
  FROM `physionet-data.mimiciv_3_1_hosp.emar` em
  JOIN `physionet-data.mimiciv_ed.edstays` e
    ON em.hadm_id = e.hadm_id
  LEFT JOIN icu_times icu
    ON em.hadm_id = icu.hadm_id
  WHERE (e.disposition = 'ADMITTED' OR e.hadm_id IS NOT NULL)
    AND em.event_txt IN (
      'Administered',
      'Partial Administered',
      'Delayed Administered',
      'Administered Bolus from IV Drip',
      'Administered in Other Location',
      'Started',
      'Restarted',
      'Stopped',
      'Stopped As Directed',
      'Stopped - Unscheduled',
      'Stopped in Other Location',
      'Rate Change'
    )
    AND CAST(em.charttime AS TIMESTAMP) < COALESCE(icu.icu_intime, TIMESTAMP('9999-01-01'))
)

SELECT * FROM pyxis_part
UNION ALL
SELECT * FROM emar_part
ORDER BY subject_id, stay_id, COALESCE(pyxis_charttime, emar_charttime)
"""

print("Running admitted meds query...")
df_admitted_meds = query_to_df(client, query_admitted_meds)
print(f"Shape: {df_admitted_meds.shape}")
print(f"Unique patients: {df_admitted_meds['subject_id'].nunique():,}")
pyxis_rows = df_admitted_meds['in_er'].sum()
emar_rows = (~df_admitted_meds['in_er']).sum()
print(f"\nED (Pyxis) rows: {pyxis_rows:,}")
print(f"Ward (eMAR) rows: {emar_rows:,}")
df_admitted_meds.head()

In [ ]:
DESCRIPTION = (
    "Medication administration records for admitted patients, covering both the ED boarding phase "
    "and the subsequent hospital ward stay (pre-ICU). Derived from hosp.emar. "
    "event_txt values mapped to four discrete actions: administer_medication, start_medication, "
    "stop_medication, rate_change. Flush, topical, and documentation events excluded."
)

ds = Dataset.from_pandas(df_admitted_meds, split='meds_admit')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="meds_admitted", data_dir="stay_medications")
push_dataset_card("ADS599-Capstone/raw_data", config_name="meds_admitted", split="meds_admit", data_dir="stay_medications", description=DESCRIPTION)
print("Pushed meds_admitted to HuggingFace Hub.")

In [ ]:
# 1. Filter to in_er=True (ED phase only) and map drug classes
#
# meds_admitted contains two populations:
#   in_er = True  (11.4%, 996,474 rows): Pyxis dispenses while the admitted patient
#               was still physically in the ED before hospital transfer, making it 
#               SAFE to use.
#   in_er = False (88.6%, 7,775,145 rows): eMAR records from the hospital ward after
#               the patient left the ED, DATA LEAKAGE and excluded entirely.
#
# Why the leakage risk is real:
#   The RL agent makes disposition decisions during the ED stay. Ward medications
#   are administered after the decision has already been made. Including them as
#   predictors would give the agent information that did not exist at decision time.
#
# EVENT_TXT filter (null event_txt INCLUDED):
#   All in_er=True rows are Pyxis dispenses with null event_txt (no EMAR record).
#   The original filter requiring 'Administered' in event_txt returned 0 records
#   because Pyxis rows are not linked to EMAR. Fix: exclude only explicit
#   non-administration values; null passes through as a valid dispense.

# Same 22-class vocabulary used in medrecon and ed_meds so features share the same space.
NAME_TO_CLASS = {
    r'morphine|hydromorphone|dilaudid|fentanyl|oxycodone|tramadol|ketorolac|toradol': 'Analgesic - Opioid/NSAID',
    r'acetaminophen|tylenol': 'Analgesic - Acetaminophen',
    r'ibuprofen|naproxen': 'Analgesic - NSAID',
    r'ondansetron|zofran|promethazine|phenergan|metoclopramide|reglan|prochlorperazine': 'Antiemetic',
    r'vancomycin|ceftriaxone|cefazolin|piperacillin|zosyn|azithromycin|ciprofloxacin|metronidazole|flagyl|ampicillin|meropenem|levofloxacin': 'Antibiotic',
    r'lorazepam|ativan|diazepam|valium|midazolam|versed|alprazolam': 'Benzodiazepine - Sedative/Anxiolytic',
    r'heparin|enoxaparin|lovenox|warfarin|coumadin|apixaban|rivaroxaban': 'Anticoagulant',
    r'metoprolol|lopressor|labetalol|atenolol|carvedilol': 'Beta Blocker',
    r'lisinopril|enalapril|captopril|ramipril': 'ACE Inhibitor',
    r'amlodipine|diltiazem|verapamil|nifedipine': 'Calcium Channel Blocker',
    r'nitroglycerin|nitro': 'Nitrate',
    r'amiodarone|adenosine|digoxin': 'Antiarrhythmic',
    r'furosemide|lasix|bumetanide|torsemide|hydrochlorothiazide': 'Diuretic',
    r'methylprednisolone|prednisone|dexamethasone|hydrocortisone': 'Corticosteroid',
    r'pantoprazole|omeprazole|famotidine|pepcid|ranitidine': 'GI - Acid Suppression',
    r'albuterol|ipratropium|atrovent|levalbuterol': 'Bronchodilator',
    r'insulin|dextrose|glucagon': 'Insulin/Glucose',
    r'haloperidol|haldol|olanzapine|quetiapine|risperidone': 'Antipsychotic',
    r'levetiracetam|keppra|phenytoin|dilantin|valproate|depakote|lacosamide': 'Anticonvulsant',
    r'normal saline|sodium chloride 0\.9|lactated|ringer|d5w|dextrose 5': 'IV Fluid',
    r'aspirin|clopidogrel|plavix|ticagrelor': 'Antiplatelet',
}

def map_name_to_class(name):
    if pd.isna(name):
        return 'Other'
    name_lower = str(name).lower()
    for pattern, drug_class in NAME_TO_CLASS.items():
        if re.search(pattern, name_lower):
            return drug_class
    return 'Other'

NON_ADMIN_VALUES = [
    'not given', 'refused', 'held', 'due', 'stopped', 'restarted', 'rate change'
]

adm_er = df_admitted_meds[df_admitted_meds['in_er'] == True].copy()
print(f'in_er = True rows: {len(adm_er):,}')
print(f'in_er = False rows excluded (leakage): {(~df_admitted_meds["in_er"]).sum():,}')

if 'event_txt' in adm_er.columns:
    non_admin_mask = adm_er['event_txt'].str.lower().str.contains(
        '|'.join(NON_ADMIN_VALUES), na = False
    )
    n_before = len(adm_er)
    adm_er = adm_er[~non_admin_mask]
    print(f'After event_txt filter: {n_before:,} -> {len(adm_er):,} '
          f'(null event_txt Pyxis rows retained)')

adm_er['medication'] = adm_er['emar_medication'].fillna(adm_er['pyxis_medication'])
adm_er['drug_class'] = adm_er['medication'].apply(map_name_to_class)
print('\nDrug class distribution (in_er = True):')
print(adm_er['drug_class'].value_counts().head(15))

In [ ]:
# 2. Compute MINUTES_INTO_STAY and build action file
#
# Action time priority: EMAR charttime > Pyxis charttime > scheduletime
#   For in_er = True rows emar_charttime is always null (Pyxis-only), so
#   pyxis_charttime is used. scheduletime used as last resort.
#
# Timezone handling:
#   Pyxis charttime carries UTC timezone labels; cohort timestamps are naive.
#   Stripped via tz_localize(None) to avoid TypeError on subtraction.
#
# JOIN KEY: stay_id -> ed_stay_id (NOT hadm_id)
#   hadm_id is not unique per ED visit and causes row explosion.
#   stay_id maps 1:1 to ed_stay_id in the cohort table.
#
# class_to_idx built here from the drug classes so the action vocabulary 
# is self-contained. Matches ed_meds action vocab and both derive from the same
# 22 class list.

# Build action vocabulary from this notebook's drug classes
class_vocab = sorted(NAME_TO_CLASS.values())
class_vocab = sorted(set(class_vocab) | {'Other'})  # include Other
class_to_idx = {c: i for i, c in enumerate(class_vocab)}

adm_er['action_time'] = (adm_er['emar_charttime']
                          .fillna(adm_er['pyxis_charttime'])
                          .fillna(adm_er.get('scheduletime')))

if adm_er['action_time'].dt.tz is not None:
    adm_er['action_time'] = adm_er['action_time'].dt.tz_localize(None)

cohort = load_dataset('ADS599-Capstone/raw_data', 'cohort_base', split = 'base',
                      verification_mode = 'no_checks').to_pandas()
cohort['ed_intime'] = pd.to_datetime(cohort['ed_intime'])

# join on stay_id = ed_stay_id; drop stay_id after to avoid ambiguity
adm_er = adm_er.merge(
    cohort[['ed_stay_id','ed_intime']],
    left_on='stay_id', right_on = 'ed_stay_id', how = 'left'
)

if adm_er['ed_intime'].dt.tz is not None:
    adm_er['ed_intime'] = adm_er['ed_intime'].dt.tz_localize(None)

adm_er['minutes_into_stay'] = (
    (adm_er['action_time'] - adm_er['ed_intime']).dt.total_seconds() / 60
)
adm_er = adm_er[
    (adm_er['minutes_into_stay'] >= 0) &
    (adm_er['minutes_into_stay'] <= 1440)
].copy()

assert len(adm_er) <= len(df_admitted_meds[df_admitted_meds['in_er'] == True]), \
    'Row count increased after cohort join, check for duplicate stay_ids'

adm_er['action_idx'] = adm_er['drug_class'].map(class_to_idx).fillna(-1).astype(int)
adm_er['source'] = 'meds_admitted'

# ed_stay_id comes from cohort merge; subject_id from original df_admitted_meds
adm_out = adm_er[[
    'ed_stay_id', 'subject_id', 'action_time', 'minutes_into_stay',
    'drug_class', 'action_idx', 'source'
]].rename(columns = {'action_time':'charttime'})

adm_out.to_csv('features_meds_admitted_er_actions.csv', index = False)
print(f'Saved features_meds_admitted_er_actions.csv  shape: {adm_out.shape}')
print(f'Final ED action records: {len(adm_out):,}')

In [ ]:
# 3. Combine with ed_only_meds ACTION FILE
#
# Both action files share the same schema so they can be concatenated directly.
# The combined file is the primary input for RL trajectory construction to be combined 
# with lab and microbiology action files on ed_stay_id and minutes_into_stay.

ed_actions = pd.read_csv('features_ed_meds_timestep_actions.csv')
all_actions = pd.concat([ed_actions, adm_out], ignore_index = True)
all_actions = all_actions.sort_values(['ed_stay_id','minutes_into_stay']).reset_index(drop = True)

all_actions.to_csv('features_all_med_actions.csv', index = False)
print(f'Saved features_all_med_actions.csv  shape: {all_actions.shape}')
print(f'  ed_only_meds actions:   {len(ed_actions):,}')
print(f'  meds_admitted actions:  {len(adm_out):,}')
print(f'  Combined:               {len(all_actions):,}')
print(f'\nSchema: ed_stay_id | subject_id | charttime | minutes_into_stay | drug_class | action_idx | source')